# Simulacija AI-vođene uvjetovane primopredaje u LEO NTN mreži
Diplomski rad: *Arhitektura ne-zemaljskih 6G NTN mreža i pokrivenost ruralnih područja*

Notebook je podijeljen prema odjeljcima poglavlja 5:
- **5.1** parametri simulacije i LEO konstelacije
- **5.2** geometrija, kanal i mobilnost korisnika
- **5.3** oznake, treniranje i vrednovanje ML modela
- **5.4** simulacijski hod, metode primopredaje i metrike

Na kraju se generiraju tablice i grafovi za poglavlja 3 i 6.
Pokretanje: *Runtime → Run all*. Cijeli notebook u Colabu traje nekoliko minuta.

In [ ]:
import time, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance

plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 110, 'savefig.dpi': 300})
BOJE = {'A3': '#2E6DA4', 'D1-CHO': '#2A9D8F', 'Rezerva': '#5A5A5A', 'AI-CHO': '#E07A2F', 'AI-CHO-U': '#7B4FA0'}
METODE = ['A3', 'D1-CHO', 'Rezerva', 'AI-CHO', 'AI-CHO-U']
# AI-CHO   = stabilna radna točka (trošak odabran kriterijem najmanje neželjenih događaja)
# AI-CHO-U = uravnotežena radna točka (najveća propusnost uz RLF i ping-pong najviše na razini A3)
OUT = 'rezultati'
os.makedirs(OUT, exist_ok=True)

## 5.1. Parametri simulacije i LEO konstelacije
Konstelacija je tipa Walker-delta *i: T/P/F* s kružnim orbitama.
Sve vrijednosti mijenjaju se u rječniku `CFG`.

In [ ]:
CFG = dict(
    # --- konstelacija (Walker-delta) ---
    h_km=600.0,          # visina orbite
    incl_deg=53.0,       # nagib orbite
    P=40,                # broj orbitalnih ravnina
    S=22,                # broj satelita po ravnini
    F=1,                 # fazni parametar
    # --- ruralno područje i korisnici ---
    lat0=44.55, lon0=15.37,   # središte ruralnog područja (Lika, Hrvatska)
    radius_km=30.0,           # polumjer područja
    n_users=20,
    frac_vehicles=0.3,        # udio korisnika u vozilu
    v_vehicle_kmh=(50, 90),
    # --- vrijeme ---
    dt=0.1,                   # korak simulacije [s]
    duration_s=1200.0,        # trajanje jednog scenarija [s]
    # --- radijski parametri (TR 38.821, LEO-600, S-pojas) ---
    fc_hz=2.0e9,
    bw_hz=30e6,
    eirp_density_dbw_mhz=34.0,
    scan_q=1.3,               # gubitak skeniranja faznog niza: -10 q log10(cos θ)
    ue_gain_dbi=0.0,
    ue_nf_db=7.0,
    pol_loss_db=3.0,
    other_loss_db=1.0,
    n_subcarriers=936,        # 78 PRB x 12 (30 MHz, 30 kHz)
    elev_min_deg=20.0,        # minimalna elevacija za vezu
    nlos_mean_s=3.0,          # prosječno trajanje zasjenjenja (NLOS) u dvostanjskom Markovljevom modelu
    tau_sf_s=3.0,             # vremenska korelacija sjenčanja
    meas_noise_db=1.0,        # pogreška mjerenja RSRP-a
    l3_alpha=0.5,             # koeficijent L3 filtra
    # --- veza i primopredaja ---
    q_out_db=-8.0, q_in_db=-6.0, t310_s=1.0,
    reest_s=1.0,              # trajanje ponovne uspostave veze nakon RLF-a
    ho_interrupt_s=0.1,       # prekid prijenosa podataka pri primopredaji
    a3_cmd_delay_s=0.1,       # kašnjenje izvješća i naredbe kod A3
    pingpong_s=3.0,           # povratak na prethodni satelit unutar ovog vremena = ping-pong
    pingpong_windows_s=(3.0, 10.0, 30.0),  # pragovi za analizu osjetljivosti ping-pong metrike
    user_bw_hz=5e6,           # pojas dodijeljen korisniku za izračun propusnosti
    # --- parametri metoda (podešavaju se u odjeljku 5.4) ---
    a3_offset_db=2.0, a3_ttt_s=0.32,
    d1_th1_km=400.0, d1_hys_km=100.0, d1_ttt_s=0.32,
    ai_decision_s=0.5, ai_window_s=20.0, ai_bonus_se=0.15, ai_margin=0.05,
    ai_fallback_snr_db=-4.0,
    n_cand=6,
    # --- poboljšanja (odjeljak 5.3) ---
    label_mode='dp',          # 'dp' = globalno optimalan niz satelita (dinamičko programiranje), 'greedy' = prozor unaprijed
    dp_ho_cost=20.0,          # trošak jedne primopredaje u pravilu označavanja [bit/s/Hz · s]; odabran validacijom (101/102)
    switch_weight=3.0,        # težina odluka o primopredaji pri treniranju (neuravnoteženost klasa)
    ai_min_dwell_s=3.0,       # najkraće zadržavanje prije nove AI odluke; namjerno kraće od pragova ping-pong metrike (10 s, 30 s)
)
RE = 6371.0; MU = 398600.4418; WE = 7.2921159e-5; C_KMS = 299792.458
N_SATS = CFG['P'] * CFG['S']
a_km = RE + CFG['h_km']
T_orb = 2*np.pi*np.sqrt(a_km**3/MU)
print(f"Satelita: {N_SATS}, ophodno vrijeme: {T_orb/60:.1f} min, brzina: {np.sqrt(MU/a_km):.2f} km/s")

## 5.2. Geometrija, kanal i mobilnost korisnika
### Položaj i brzina satelita
Kružna orbita u inercijalnom sustavu (ECI), zatim rotacija u sustav vezan uz Zemlju (ECEF).

In [ ]:
def walker_elements(cfg):
    P, S, F = cfg['P'], cfg['S'], cfg['F']
    p = np.repeat(np.arange(P), S); s = np.tile(np.arange(S), P)
    raan = 2*np.pi*p/P
    m0 = 2*np.pi*s/S + 2*np.pi*F*p/(P*S)
    return raan, m0

RAAN, M0 = walker_elements(CFG)
INC = np.radians(CFG['incl_deg']); N_MOT = np.sqrt(MU/a_km**3)

def sat_state_ecef(t, idx=None):
    """Položaj [km] i brzina [km/s] satelita u ECEF sustavu u trenutku t."""
    raan = RAAN if idx is None else RAAN[idx]; m0 = M0 if idx is None else M0[idx]
    u = m0 + N_MOT*t
    cO, sO, cu, su, ci, si = np.cos(raan), np.sin(raan), np.cos(u), np.sin(u), np.cos(INC), np.sin(INC)
    r = a_km*np.stack([cO*cu - sO*su*ci, sO*cu + cO*su*ci, su*si], -1)
    v = a_km*N_MOT*np.stack([-cO*su - sO*cu*ci, -sO*su + cO*cu*ci, cu*si], -1)
    th = WE*t; c, s = np.cos(th), np.sin(th)
    R = np.array([[c, s, 0], [-s, c, 0], [0, 0, 1]])
    r_e = r @ R.T; v_e = v @ R.T
    v_e = v_e - np.cross(np.array([0, 0, WE]), r_e)
    return r_e, v_e

def geodetic_to_ecef(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return RE*np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)], -1)

def geometry(r_s, v_s, r_u, v_u):
    """Elevacija [°], kosa udaljenost [km], radijalna brzina [km/s], kut od nadira satelita [°],
    udaljenost korisnika od podsatelitske točke [km]. Oblici: r_u (U,3), r_s (S,3) -> (U,S)."""
    rho = r_s[None] - r_u[:, None]
    d = np.linalg.norm(rho, axis=-1)
    up = r_u/np.linalg.norm(r_u, axis=-1, keepdims=True)
    elev = np.degrees(np.arcsin(np.clip(np.einsum('usk,uk->us', rho, up)/d, -1, 1)))
    rr = np.einsum('usk,usk->us', rho, v_s[None] - v_u[:, None])/d
    nad = -r_s/np.linalg.norm(r_s, axis=-1, keepdims=True)
    off = np.degrees(np.arccos(np.clip(np.einsum('usk,sk->us', -rho, nad)/d, -1, 1)))
    cosg = np.clip(up @ (r_s/np.linalg.norm(r_s, axis=-1, keepdims=True)).T, -1, 1)
    nadir_km = RE*np.arccos(cosg)
    return elev, d, rr, off, nadir_km

### Mobilnost korisnika
Korisnici su nasumično raspoređeni unutar ruralnog područja. Dio korisnika miruje,
a dio se kreće vozilom stalnom brzinom uz povremene promjene smjera.

In [ ]:
def make_users(cfg, rng):
    U = cfg['n_users']
    rr = cfg['radius_km']*np.sqrt(rng.random(U)); ang = rng.random(U)*2*np.pi
    east, north = rr*np.cos(ang), rr*np.sin(ang)
    moving = rng.random(U) < cfg['frac_vehicles']
    speed = np.where(moving, rng.uniform(*cfg['v_vehicle_kmh'], U)/3600.0, 0.0)  # km/s
    heading = rng.random(U)*2*np.pi
    return dict(east=east, north=north, speed=speed, heading=heading, moving=moving)

def users_step(us, dt, rng):
    turn = (rng.random(len(us['east'])) < dt/60.0) & us['moving']   # promjena smjera ~ jednom u minuti
    us['heading'] = np.where(turn, rng.random(len(turn))*2*np.pi, us['heading'])
    us['east'] = us['east'] + us['speed']*np.cos(us['heading'])*dt
    us['north'] = us['north'] + us['speed']*np.sin(us['heading'])*dt

def users_ecef(us, cfg):
    lat = cfg['lat0'] + np.degrees(us['north']/RE)
    lon = cfg['lon0'] + np.degrees(us['east']/(RE*np.cos(np.radians(cfg['lat0']))))
    r = geodetic_to_ecef(lat, lon)
    la, lo = np.radians(lat), np.radians(lon)
    e_hat = np.stack([-np.sin(lo), np.cos(lo), np.zeros_like(lo)], -1)
    n_hat = np.stack([-np.sin(la)*np.cos(lo), -np.sin(la)*np.sin(lo), np.cos(la)], -1)
    v = (us['speed']*np.cos(us['heading']))[:, None]*e_hat + (us['speed']*np.sin(us['heading']))[:, None]*n_hat
    return r, v

### Model kanala (TR 38.811, ruralno okruženje, S-pojas)
Primljena snaga: EIRP − gubitak skeniranja − FSPL − (gušenje okoline u NLOS) − sjenčanje − ostali gubici.
Vjerojatnost LOS-a, standardna devijacija sjenčanja i gušenje okoline (*clutter loss*) ovise o elevaciji
i interpoliraju se iz tablica TR 38.811 za ruralno okruženje (vrijednosti provjeriti u izvoru, Tablice 6.6.1-1 i 6.6.2-3).
Stanje LOS/NLOS za svaki par korisnik–satelit modelira se **dvostanjskim Markovljevim lancem**: zasjenjenje
u prosjeku traje `nlos_mean_s` sekundi, a vjerojatnost prijelaza LOS → NLOS bira se tako da udio vremena u LOS
stanju odgovara vjerojatnosti LOS-a iz TR 38.811 za trenutnu elevaciju. Sjenčanje je vremenski korelirano
autoregresijskim procesom prvog reda (`tau_sf_s`).

In [ ]:
EL_TAB = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90], float)
PLOS_TAB = np.array([78.2, 86.9, 91.9, 92.9, 93.5, 94.0, 94.9, 95.2, 99.8])/100
SF_LOS_TAB = np.array([1.79, 1.14, 1.14, 0.92, 1.42, 1.56, 0.85, 0.72, 0.72])
SF_NLOS_TAB = np.array([8.93, 9.08, 8.78, 10.25, 10.56, 10.74, 10.17, 11.52, 11.52])
CL_TAB = np.array([19.52, 18.17, 18.42, 18.28, 18.63, 17.68, 16.50, 16.30, 16.30])

def fspl_db(d_km, fc_hz):
    return 32.45 + 20*np.log10(fc_hz/1e6) + 20*np.log10(d_km)

EIRP_DBM = CFG['eirp_density_dbw_mhz'] + 10*np.log10(CFG['bw_hz']/1e6) + 30
NOISE_DBM = -174 + CFG['ue_nf_db'] + 10*np.log10(CFG['bw_hz'])
print(f"EIRP = {EIRP_DBM:.1f} dBm, šum = {NOISE_DBM:.1f} dBm")

def from_normal(x):
    from scipy.special import ndtr
    return ndtr(x)

### Generiranje kanalnog traga
Za svaki scenarij najprije se izračuna kompletan kanalni trag: geometrija i SNR za svaki par
korisnik–satelit u svakom koraku. Sve metode primopredaje zatim se izvode nad **istim tragom**,
pa razlike u rezultatima proizlaze isključivo iz metode, a ne iz slučajnosti kanala.

In [ ]:
def generate_trace(cfg, seed, verbose=True):
    rng = np.random.default_rng(seed)
    t0 = rng.uniform(0, T_orb)                   # slučajan početni položaj konstelacije
    dt, T = cfg['dt'], int(cfg['duration_s']/cfg['dt'])
    us = make_users(cfg, rng)
    # 1) relevantni sateliti: oni koji tijekom scenarija prođu iznad (elev_min - 5°)
    coarse = np.arange(0, cfg['duration_s']+1, 5.0)
    r_c = geodetic_to_ecef(np.array([cfg['lat0']]), np.array([cfg['lon0']]))
    rel = np.zeros(N_SATS, bool)
    for tc in coarse:
        rs, vs = sat_state_ecef(t0+tc)
        e, *_ = geometry(rs, vs, r_c, np.zeros_like(r_c))
        rel |= e[0] > cfg['elev_min_deg'] - 8
    sid = np.where(rel)[0]; S = len(sid); U = cfg['n_users']
    shp = (T, U, S)
    tr = {k: np.zeros(shp, np.float32) for k in ['elev', 'snr', 'rr', 'nadir', 'rsrp_meas']}
    tr['users_xy'] = np.zeros((T, U, 2), np.float32)
    rho_s = np.exp(-dt/cfg['tau_sf_s'])
    x_sf = rng.standard_normal((U, S))
    p_nl = 1 - np.exp(-dt/cfg['nlos_mean_s'])              # vjerojatnost izlaska iz NLOS u jednom koraku
    los = None
    filt = None
    for k in range(T):
        t = t0 + k*dt
        rs, vs = sat_state_ecef(t, sid)
        ru, vu = users_ecef(us, cfg)
        elev, d, rr, off, nad = geometry(rs, vs, ru, vu)
        e_c = np.clip(elev, 10, 90)
        p_los = np.interp(e_c, EL_TAB, PLOS_TAB)
        if los is None:
            los = rng.random((U, S)) < p_los                # početno stanje iz stacionarne raspodjele
        else:
            p_ln = np.minimum(p_nl*(1 - p_los)/p_los, 1.0)   # LOS → NLOS, čuva stacionarni udio p_los
            u_ = rng.random((U, S))
            los = np.where(los, u_ >= p_ln, u_ < p_nl)
        sf_sig = np.where(los, np.interp(e_c, EL_TAB, SF_LOS_TAB), np.interp(e_c, EL_TAB, SF_NLOS_TAB))
        cl = np.where(los, 0.0, np.interp(e_c, EL_TAB, CL_TAB))
        scan = -10*cfg['scan_q']*np.log10(np.clip(np.cos(np.radians(off)), 0.05, 1))
        prx = (EIRP_DBM - scan - fspl_db(d, cfg['fc_hz']) - cl - sf_sig*x_sf
               - cfg['pol_loss_db'] - cfg['other_loss_db'] + cfg['ue_gain_dbi'])
        snr = prx - NOISE_DBM
        snr = np.where(elev > 0, snr, -50.0)
        rsrp = prx - 10*np.log10(cfg['n_subcarriers'])
        meas = rsrp + cfg['meas_noise_db']*rng.standard_normal((U, S))
        filt = meas if filt is None else (1-cfg['l3_alpha'])*filt + cfg['l3_alpha']*meas
        tr['elev'][k], tr['snr'][k], tr['rr'][k], tr['nadir'][k], tr['rsrp_meas'][k] = elev, snr, rr, nad, filt
        tr['users_xy'][k] = np.stack([us['east'], us['north']], -1)
        x_sf = rho_s*x_sf + np.sqrt(1-rho_s**2)*rng.standard_normal((U, S))
        users_step(us, dt, rng)
    tr.update(sid=sid, t0=t0, T=T, U=U, S=S, dt=dt, moving=us['moving'])
    # preostalo vrijeme vidljivosti (iz efemerida: buduća elevacija je poznata unaprijed)
    vis = tr['elev'] >= cfg['elev_min_deg']
    rem = np.zeros(shp, np.float32); run = np.zeros((U, S), np.float32)
    for k in range(T-1, -1, -1):
        run = np.where(vis[k], run + dt, 0.0); rem[k] = run
    tr['rem_vis'] = rem
    if verbose:
        nvis = vis.sum(-1)
        print(f"scenarij {seed}: {S} relevantnih satelita, vidljivo iznad {cfg['elev_min_deg']:.0f}°: "
              f"prosječno {nvis.mean():.1f}, najmanje {nvis.min()}")
    return tr

## 5.4. Simulacijski hod i metode primopredaje
Zajednički simulator veze za svakog korisnika i svaki korak:
- računa SNR i propusnost na poslužnom satelitu,
- prati uvjet prekida veze (RLF): SNR < Q_out dulje od T310,
- primjenjuje odluku metode primopredaje i bilježi primopredaje, neuspjele primopredaje (HOF) i ping-pong.

Metode:
- **A3** – klasična primopredaja temeljena na mjerenjima (susjed jači za offset tijekom TTT; naredba kasni),
- **D1-CHO** – uvjetovana primopredaja s lokacijskim okidačem (Release 17) i istim rezervnim A3 okidačem kao AI-CHO,
- **AI-CHO** – uvjetovana primopredaja vođena ML modelom (odjeljak 4.4),
- **AI-CHO-U** – isti sustav kao AI-CHO, ali s modelom istreniranim uz manji trošak primopredaje (uravnotežena radna točka),
- **Rezerva** – ablacija: isti sustav kao AI-CHO, ali model nikad ne predlaže primopredaju, pa radi samo rezervni A3 okidač.
  Razlika AI-CHO − Rezerva pokazuje koliko doprinosi sam model strojnog učenja.

In [ ]:
def throughput_bps(snr_db, bw):
    se = np.minimum(0.6*np.log2(1 + 10**(snr_db/10)), 4.4)
    return np.where(snr_db < -10, 0.0, se*bw)

class Link:
    """Stanje veze jednog korisnika, zajedničko svim metodama."""
    def __init__(self, cfg, tr, u):
        self.cfg, self.tr, self.u = cfg, tr, u
        self.serv = None; self.prev = None; self.t_last_ho = -1e9
        self.below = 0.0; self.outage_until = -1.0
        self.n_ho = self.n_pp = self.n_hof = self.n_rlf = 0
        self.tp_sum = 0.0; self.snr_log = []; self.ho_times = []; self.ho_events = []; self.serv_log = np.full(tr['T'], -1, np.int32)

    def connectable(self, k):
        return self.tr['elev'][k, self.u] >= self.cfg['elev_min_deg']

    def best_by_rsrp(self, k):
        r = np.where(self.connectable(k), self.tr['rsrp_meas'][k, self.u], -np.inf)
        return int(np.argmax(r)) if np.isfinite(r.max()) else None

    def handover(self, k, target, src='pravilo'):
        t = k*self.cfg['dt']; snr_t = self.tr['snr'][k, self.u, target]
        if snr_t < self.cfg['q_out_db'] or not self.connectable(k)[target]:
            self.n_hof += 1; self.rlf(k); return
        if self.prev == target and t - self.t_last_ho < self.cfg['pingpong_s']:
            self.n_pp += 1
        self.ho_events.append((t, self.serv, target, src))   # src: 'AI', 'rezerva' ili 'pravilo'
        self.prev, self.serv, self.t_last_ho = self.serv, target, t
        self.n_ho += 1; self.ho_times.append(t)
        self.outage_until = max(self.outage_until, t + self.cfg['ho_interrupt_s'])

    def rlf(self, k):
        self.n_rlf += 1; self.serv = None; self.below = 0.0
        self.outage_until = k*self.cfg['dt'] + self.cfg['reest_s']

    def step(self, k):
        """Vraća True ako je korisnik povezan i može primijeniti odluku metode."""
        cfg, t = self.cfg, k*self.cfg['dt']
        if self.serv is None:
            if t >= self.outage_until:
                b = self.best_by_rsrp(k)
                if b is not None and self.tr['snr'][k, self.u, b] > cfg['q_in_db']:
                    self.serv, self.prev = b, None
            self.snr_log.append(np.nan); return False
        snr = self.tr['snr'][k, self.u, self.serv]
        if not self.connectable(k)[self.serv]:
            snr = min(snr, cfg['q_out_db'] - 1)
        self.serv_log[k] = self.serv
        self.below = self.below + cfg['dt'] if snr < cfg['q_out_db'] else (0.0 if snr > cfg['q_in_db'] else self.below)
        if self.below >= cfg['t310_s']:
            self.rlf(k); self.snr_log.append(np.nan); return False
        if t >= self.outage_until:
            self.tp_sum += throughput_bps(snr, cfg['user_bw_hz'])*cfg['dt']
        self.snr_log.append(snr)
        return True

    def pingpong_count(self, W):
        """Broj primopredaja a→b kojima je prethodila primopredaja b→a unutar W sekundi."""
        ev = self.ho_events
        return sum(1 for i in range(1, len(ev)) if ev[i][1] == ev[i-1][2] and ev[i][2] == ev[i-1][1]
                   and ev[i][0] - ev[i-1][0] < W)

    def results(self):
        dur_min = self.tr['T']*self.cfg['dt']/60
        snr = np.array(self.snr_log, float)
        pp = {f'pp{int(W)}_pct': 100*self.pingpong_count(W)/max(self.n_ho, 1) for W in self.cfg['pingpong_windows_s']}
        return dict(**pp, ho_per_min=self.n_ho/dur_min, rlf_per_min=self.n_rlf/dur_min, hof_per_min=self.n_hof/dur_min,
                    pingpong_pct=100*self.n_pp/max(self.n_ho, 1), tp_mbps=self.tp_sum/(self.tr['T']*self.cfg['dt'])/1e6,
                    outage_pct=100*np.mean(~np.isfinite(snr) | (snr < self.cfg['q_out_db'])),
                    mean_snr_db=np.nanmean(snr))

def run_a3(cfg, tr, u):
    L = Link(cfg, tr, u); ttt = 0.0; cand = None; pending = None
    for k in range(tr['T']):
        if not L.step(k): ttt, cand, pending = 0.0, None, None; continue
        if pending is not None and k >= pending[0]:
            L.handover(k, pending[1]); pending = None; ttt = 0.0; continue
        if pending is not None: continue
        r = np.where(L.connectable(k), tr['rsrp_meas'][k, u], -np.inf)
        rs = r[L.serv]; r[L.serv] = -np.inf; b = int(np.argmax(r))
        if np.isfinite(r[b]) and r[b] > rs + cfg['a3_offset_db']:
            ttt = ttt + cfg['dt'] if b == cand else cfg['dt']; cand = b
        else:
            ttt, cand = 0.0, None
        if ttt >= cfg['a3_ttt_s']:
            pending = (k + int(round(cfg['a3_cmd_delay_s']/cfg['dt'])), cand); ttt = 0.0
    return L

def a3_fallback(cfg, tr, u, k, L, st):
    """Rezervni okidač (A3) koji koriste obje varijante CHO kad je SNR poslužnog satelita slab.
    st = [ttt, kandidat]; vraća True ako je izvedena primopredaja."""
    if tr['snr'][k, u, L.serv] >= cfg['ai_fallback_snr_db']:
        st[0], st[1] = 0.0, None; return False
    r = np.where(L.connectable(k), tr['rsrp_meas'][k, u], -np.inf)
    rs = r[L.serv]; r[L.serv] = -np.inf; b = int(np.argmax(r))
    if np.isfinite(r[b]) and r[b] > rs + cfg['a3_offset_db']:
        st[0] = st[0] + cfg['dt'] if b == st[1] else cfg['dt']; st[1] = b
    else:
        st[0], st[1] = 0.0, None
    if st[0] >= cfg['a3_ttt_s']:
        L.handover(k, st[1], src='rezerva'); st[0], st[1] = 0.0, None; return True
    return False

def run_d1(cfg, tr, u):
    L = Link(cfg, tr, u); ttt = 0.0; cand = None; fb = [0.0, None]
    for k in range(tr['T']):
        if not L.step(k): ttt, cand = 0.0, None; continue
        nad = np.where(L.connectable(k), tr['nadir'][k, u], np.inf)
        ds = nad[L.serv]; nad[L.serv] = np.inf; b = int(np.argmin(nad))
        if ds > cfg['d1_th1_km'] and nad[b] < ds - cfg['d1_hys_km']:
            ttt = ttt + cfg['dt'] if b == cand else cfg['dt']; cand = b
        else:
            ttt, cand = 0.0, None
        if ttt >= cfg['d1_ttt_s']:
            L.handover(k, cand); ttt = 0.0; continue
        a3_fallback(cfg, tr, u, k, L, fb)
    return L

## 5.3. Oznake, značajke i treniranje ML modela
**Oznaka** (idealna odluka) računa se iz budućeg dijela traga koji simulator poznaje. Zadani način (`label_mode='dp'`)
je Viterbijev algoritam: za cijeli scenarij traži se niz poslužnih satelita koji maksimizira ukupnu spektralnu
učinkovitost umanjenu za trošak `dp_ho_cost` po svakoj primopredaji. Takve oznake sadrže znatno manje ping-pong
primopredaja od pohlepnog pravila. Odluke o primopredaji pri treniranju dobivaju težinu `switch_weight`,
jer ih je oko 50 puta manje od odluka „ostani”.

Alternativni način (`label_mode='greedy'`): za svakog kandidata
računa se prosječna spektralna učinkovitost (bit/s/Hz) u idućih `ai_window_s` sekundi; ispod minimalne
elevacije ona je 0. Poslužni satelit dobiva prednost `ai_bonus_se`, čime pravilo označavanja kažnjava
nepotrebne primopredaje (jedna primopredaja mora donijeti barem toliko bolju budućnost).
Kandidat čiji je trenutni SNR ispod Q_in ne može biti oznaka.

**Značajke** (Tablica 4.2) koriste samo podatke dostupne u trenutku odluke: geometriju iz efemerida
i filtrirana mjerenja RSRP-a.

In [ ]:
FEATURES = ['elev', 'elev_rate', 'slant_km', 'range_rate', 'rsrp', 'rsrp_trend', 'rsrp_minus_serv',
            'rem_vis', 'serv_rem_vis', 'serv_rsrp_trend', 'is_serving', 'is_prev_serving', 't_since_ho']

def candidates(cfg, tr, u, k, serv):
    r = np.where(tr['elev'][k, u] >= cfg['elev_min_deg'], tr['rsrp_meas'][k, u], -np.inf)
    order = [int(i) for i in np.argsort(-r)[:cfg['n_cand']] if np.isfinite(r[i])]
    if serv is not None and serv not in order: order.append(serv)
    return order

def features(cfg, tr, u, k, serv, cand, L=None):
    kp = max(k - int(1.0/cfg['dt']), 0)
    e, ep = tr['elev'][k, u, cand], tr['elev'][kp, u, cand]
    slant = (np.sqrt((RE*np.sin(np.radians(e)))**2 + cfg['h_km']**2 + 2*cfg['h_km']*RE) - RE*np.sin(np.radians(e)))
    rs = tr['rsrp_meas'][k, u, serv] if serv is not None else np.nan
    srv_rem = tr['rem_vis'][k, u, serv] if serv is not None else 0.0
    srv_tr = (tr['rsrp_meas'][k, u, serv] - tr['rsrp_meas'][kp, u, serv]) if serv is not None else 0.0
    return np.column_stack([e, e - ep, slant, tr['rr'][k, u, cand], tr['rsrp_meas'][k, u, cand],
                            tr['rsrp_meas'][k, u, cand] - tr['rsrp_meas'][kp, u, cand],
                            tr['rsrp_meas'][k, u, cand] - rs, tr['rem_vis'][k, u, cand],
                            np.full(len(cand), srv_rem), np.full(len(cand), srv_tr),
                            np.array([c == serv for c in cand], float),
                            np.array([L is not None and c == L.prev for c in cand], float),
                            np.full(len(cand), min(k*cfg['dt'] - L.t_last_ho, 60.0) if L is not None else 60.0)])

def oracle_choice(cfg, tr, u, k, serv, cand):
    W = int(cfg['ai_window_s']/cfg['dt'])
    fut = tr['snr'][k:k+W, u][:, cand]
    vis = tr['elev'][k:k+W, u][:, cand] >= cfg['elev_min_deg']
    se = np.where(vis, throughput_bps(fut, 1.0), 0.0)          # spektralna učinkovitost [bit/s/Hz]
    score = se.mean(0) + np.array([cfg['ai_bonus_se'] if c == serv else 0.0 for c in cand])
    score = np.where(tr['snr'][k, u, cand] > cfg['q_in_db'], score, -np.inf)
    return int(np.argmax(score))

def dp_path(cfg, tr, u):
    """Globalno optimalan niz poslužnih satelita za korisnika u (Viterbijev algoritam).
    Nagrada u svakom trenutku odluke je spektralna učinkovitost na satelitu, a svaka promjena satelita
    košta dp_ho_cost. Vraća indeks satelita za svaki trenutak odluke."""
    step = int(cfg['ai_decision_s']/cfg['dt']); ks = np.arange(0, tr['T'], step)
    snr = tr['snr'][ks, u]; ok = (tr['elev'][ks, u] >= cfg['elev_min_deg']) & (snr > cfg['q_out_db'])
    R = np.where(ok, throughput_bps(snr, 1.0), -5.0)*cfg['ai_decision_s']
    n, S = R.shape; V = R[0].copy(); back = np.zeros((n, S), np.int32)
    for i in range(1, n):
        j = int(np.argmax(V)); sw = V[j] - cfg['dp_ho_cost']
        stay = V >= sw
        back[i] = np.where(stay, np.arange(S), j)
        V = np.where(stay, V, sw) + R[i]
    path = np.zeros(n, np.int32); path[-1] = int(np.argmax(V))
    for i in range(n-1, 0, -1): path[i-1] = back[i, path[i]]
    return ks, path

def collect_dataset(cfg, tr):
    """Prolazi tragom slijedeći idealne odluke i bilježi značajke, oznake i težine."""
    X, y, grp, w = [], [], [], []
    step = int(cfg['ai_decision_s']/cfg['dt']); W = int(cfg['ai_window_s']/cfg['dt'])
    for u in range(tr['U']):
        L = Link(cfg, tr, u)
        if cfg['label_mode'] == 'dp':
            ks, path = dp_path(cfg, tr, u); target_at = dict(zip(ks.tolist(), path.tolist()))
        for k in range(tr['T']):
            if not L.step(k) or k % step or k + W >= tr['T']: continue
            cand = candidates(cfg, tr, u, k, L.serv)
            if cfg['label_mode'] == 'dp':
                tgt = target_at[k]; j = cand.index(tgt) if tgt in cand else cand.index(L.serv)
            else:
                j = oracle_choice(cfg, tr, u, k, L.serv, cand)
            X.append(features(cfg, tr, u, k, L.serv, cand, L)); lab = np.zeros(len(cand)); lab[j] = 1
            y.append(lab); grp.append(np.full(len(cand), len(grp)))
            w.append(np.full(len(cand), cfg['switch_weight'] if cand[j] != L.serv else 1.0))
            if cand[j] != L.serv: L.handover(k, cand[j])
    return np.vstack(X), np.concatenate(y), np.concatenate(grp), np.concatenate(w)

def run_ai_all(cfg, tr, model, timing=None):
    """AI-CHO za sve korisnike istovremeno: u svakom trenutku odluke model se poziva jednom za sve korisnike."""
    U = tr['U']; step = int(cfg['ai_decision_s']/cfg['dt'])
    Ls = [Link(cfg, tr, u) for u in range(U)]
    fb = [[0.0, None] for _ in range(U)]
    for k in range(tr['T']):
        active = [u for u in range(U) if Ls[u].step(k)]
        decided = set()
        if k % step == 0 and active:
            cands = [candidates(cfg, tr, u, k, Ls[u].serv) for u in active]
            Xf = np.vstack([features(cfg, tr, u, k, Ls[u].serv, c, Ls[u]) for u, c in zip(active, cands)])
            t1 = time.perf_counter(); p = model.predict_proba(Xf)[:, 1]
            if timing is not None: timing.append((time.perf_counter() - t1)/len(active))
            i0 = 0
            for u, c in zip(active, cands):
                pu = p[i0:i0+len(c)]; i0 += len(c)
                j = int(np.argmax(pu)); ps = pu[c.index(Ls[u].serv)]
                dwell_ok = k*cfg['dt'] - Ls[u].t_last_ho >= cfg['ai_min_dwell_s']
                if dwell_ok and c[j] != Ls[u].serv and pu[j] - ps > cfg['ai_margin']:
                    Ls[u].handover(k, c[j], src='AI'); decided.add(u)
        for u in active:
            if u not in decided and Ls[u].serv is not None:
                a3_fallback(cfg, tr, u, k, Ls[u], fb[u])
    return Ls

class StayModel:
    """Ablacija: model koji nikad ne predlaže primopredaju (uvijek bira poslužni satelit).
    AI-CHO s ovim modelom = isti sustav bez strojnog učenja, sve primopredaje izvodi rezervni A3 okidač."""
    def predict_proba(self, X):
        p = X[:, FEATURES.index('is_serving')]; return np.column_stack([1 - p, p])

def evaluate(cfg, tr, method, model=None, timing=None, return_links=False):
    if method == 'Rezerva':
        Ls = run_ai_all(cfg, tr, StayModel())
    elif method.startswith('AI-CHO'):
        Ls = run_ai_all(cfg, tr, model, timing)
    else:
        Ls = [(run_a3 if method == 'A3' else run_d1)(cfg, tr, u) for u in range(tr['U'])]
    df = pd.DataFrame([dict(user=u, moving=bool(tr['moving'][u]), **L.results()) for u, L in enumerate(Ls)])
    return (df, Ls) if return_links else df

def upari_ai_a3(Ls_ai, Ls_a3, seed, prozor_s=60.0):
    """Za svaku primopredaju koju je odlučio model (src='AI') traži istu primopredaju (isti izvorni i ciljni
    satelit, isti korisnik) kod A3 unutar ±prozor_s. Vraća popis parova s vremenima obiju odluka."""
    par = []
    for u, (La, Lb) in enumerate(zip(Ls_ai, Ls_a3)):
        ev_a3 = [e for e in Lb.ho_events if e[1] is not None]
        for t_ai, a, b, src in La.ho_events:
            if src != 'AI' or a is None: continue
            kand = [e[0] for e in ev_a3 if e[1] == a and e[2] == b and abs(e[0] - t_ai) <= prozor_s]
            par.append(dict(seed=seed, korisnik=u, izvor=a, cilj=b, t_ai=t_ai,
                            t_a3=min(kand, key=lambda x: abs(x - t_ai)) if kand else np.nan))
    return par

## Pokretanje: generiranje scenarija
Dva scenarija služe za podešavanje konvencionalnih metoda i treniranje modela, a deset neovisnih
scenarija (drugi raspored korisnika, drugi početni položaj konstelacije) za vrednovanje.

In [ ]:
TRAIN_SEEDS = [101, 102]
TEST_SEEDS = list(range(201, 211))   # 10 neovisnih testnih scenarija
t_start = time.time()
train_traces = [generate_trace(CFG, s) for s in TRAIN_SEEDS]
print(f"gotovo za {time.time()-t_start:.0f} s")

### Podešavanje konvencionalnih metoda
Da bi usporedba bila poštena, parametri metoda A3 i D1-CHO biraju se na scenariju za treniranje.
Kriterij je kao kod optimizacije robusnosti mobilnosti (MRO): najmanji ukupni broj neželjenih događaja
po korisniku u minuti, tj. zbroj prekida veze (RLF), neuspjelih primopredaja (HOF) i ping-pong primopredaja
(povratak na prethodni satelit unutar 10 s). Isti kriterij koristi se i za odabir postavki AI-CHO.

In [ ]:
def tune(method, grid, keys, tr):
    rows = []
    for vals in grid:
        cfg = dict(CFG, **dict(zip(keys, vals)))
        r = evaluate(cfg, tr, method).mean(numeric_only=True)
        bad = r['rlf_per_min'] + r['hof_per_min'] + r['pp10_pct']/100*r['ho_per_min']
        rows.append(dict(zip(keys, vals), **r[['ho_per_min', 'rlf_per_min', 'hof_per_min', 'pingpong_pct', 'tp_mbps']].to_dict(),
                         nezeljeni_po_min=bad))
    df = pd.DataFrame(rows).sort_values('nezeljeni_po_min')
    return df

grid_a3 = [(o, t) for o in [2.0, 4.0, 6.0, 8.0, 10.0] for t in [0.16, 0.32, 0.64, 1.28]]
tab_a3 = tune('A3', grid_a3, ['a3_offset_db', 'a3_ttt_s'], train_traces[0])
grid_d1 = [(a, b) for a in [300.0, 400.0, 500.0, 600.0] for b in [50.0, 100.0, 200.0]]
tab_d1 = tune('D1-CHO', grid_d1, ['d1_th1_km', 'd1_hys_km'], train_traces[0])
CFG['a3_offset_db'], CFG['a3_ttt_s'] = tab_a3.iloc[0][['a3_offset_db', 'a3_ttt_s']]
CFG['d1_th1_km'], CFG['d1_hys_km'] = tab_d1.iloc[0][['d1_th1_km', 'd1_hys_km']]
print("A3:", CFG['a3_offset_db'], "dB,", CFG['a3_ttt_s'], "s   D1:", CFG['d1_th1_km'], "km,", CFG['d1_hys_km'], "km")
tab_a3.round(3).to_csv(f'{OUT}/podesavanje_A3.csv', index=False)
tab_d1.round(3).to_csv(f'{OUT}/podesavanje_D1.csv', index=False)
display(tab_a3.head(5).round(3)); display(tab_d1.head(5).round(3))

### 5.3. Skup podataka i treniranje modela

In [ ]:
t1 = time.time()
parts = [collect_dataset(CFG, tr) for tr in train_traces]
X_train = np.vstack([p[0] for p in parts]); y_train = np.concatenate([p[1] for p in parts])
w_train = np.concatenate([p[3] for p in parts])
print(f"skup za treniranje: {X_train.shape[0]} redaka ({int(y_train.sum())} odluka), {time.time()-t1:.0f} s")

MODEL_PARAMS = dict(max_iter=300, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=40,
                    l2_regularization=1.0, early_stopping=True, validation_fraction=0.1, random_state=0)
model = HistGradientBoostingClassifier(**MODEL_PARAMS)
t1 = time.time(); model.fit(X_train, y_train, sample_weight=w_train); fit_s = time.time() - t1
print(f"treniranje: {fit_s:.1f} s, broj stabala: {model.n_iter_}")

### Kompromis stabilnost – propusnost (za odjeljak 6.4)
Trošak primopredaje `dp_ho_cost` u pravilu označavanja određuje koliko je AI-CHO „oprezan”.
Za nekoliko vrijednosti troška model se trenira na scenariju 101 i vrednuje na scenariju 102 (validacija),
pa se rezultati uspoređuju s A3, D1-CHO i ablacijom na istom scenariju. Testni scenariji ovdje se ne koriste.
Iz rezultata se biraju dvije radne točke, obje samo na validacijskim podacima:
- **AI-CHO (stabilna)** – `CFG['dp_ho_cost']`, najmanje neželjenih događaja (isti kriterij kao za A3 i D1-CHO),
- **AI-CHO-U (uravnotežena)** – najveća propusnost uz uvjet da su prekidi veze (RLF) i ping-pong (10 s)
  najviše na razini metode A3 na istom scenariju.
Druga točka dodana je jer prvi kriterij ne uzima u obzir propusnost; obje se vrednuju na svim testnim scenarijima.

In [ ]:
KOMPROMIS_TROSKOVI = [2.0, 5.0, 10.0, 20.0, 40.0]
tr_fit, tr_val = train_traces
rows = []
for m in ['A3', 'D1-CHO', 'Rezerva']:
    r = evaluate(CFG, tr_val, m).mean(numeric_only=True)
    rows.append(dict(metoda=m, trosak=np.nan, **r.to_dict()))
for c in KOMPROMIS_TROSKOVI:
    cfg_c = dict(CFG, dp_ho_cost=c)
    Xc, yc, _, wc = collect_dataset(cfg_c, tr_fit)
    mdl_c = HistGradientBoostingClassifier(**MODEL_PARAMS).fit(Xc, yc, sample_weight=wc)
    r = evaluate(cfg_c, tr_val, 'AI-CHO', mdl_c).mean(numeric_only=True)
    rows.append(dict(metoda='AI-CHO', trosak=c, **r.to_dict())); print(f'trošak {c:>4.0f}: gotovo', flush=True)
KOMP = pd.DataFrame(rows)
KOMP['nezeljeni_po_min'] = KOMP.rlf_per_min + KOMP.hof_per_min + KOMP.pp10_pct/100*KOMP.ho_per_min
kol = ['metoda', 'trosak', 'ho_per_min', 'pp10_pct', 'rlf_per_min', 'outage_pct', 'tp_mbps', 'nezeljeni_po_min']
KOMP[kol].round(3).to_csv(f'{OUT}/tablica_6_4_kompromis.csv', index=False)
print(KOMP[kol].round(3).to_string(index=False))

# odabir uravnotežene radne točke: najveća propusnost uz RLF i ping-pong (10 s) najviše kao kod A3
a3v = KOMP[KOMP.metoda == 'A3'].iloc[0]
ok = KOMP[(KOMP.metoda == 'AI-CHO') & (KOMP.rlf_per_min <= a3v.rlf_per_min + 1e-9) & (KOMP.pp10_pct <= a3v.pp10_pct + 1e-9)]
CFG['dp_ho_cost_U'] = float(ok.sort_values('tp_mbps').iloc[-1].trosak) if len(ok) else CFG['dp_ho_cost']
print(f"stabilna točka: trošak {CFG['dp_ho_cost']:.0f};  uravnotežena točka: trošak {CFG['dp_ho_cost_U']:.0f}")

In [ ]:
# Slika 6.3: tri male ploče s troškom primopredaje na osi x (logaritamska skala).
# AI-CHO je krivulja kroz vrijednosti troška (obrubljena točka = odabrana vrijednost);
# A3, D1-CHO i ablacija ne ovise o trošku, pa su prikazani kao vodoravne referentne linije.
ai = KOMP[KOMP.metoda == 'AI-CHO'].sort_values('trosak'); ref = KOMP[KOMP.metoda != 'AI-CHO'].set_index('metoda')
STIL = {'A3': '--', 'D1-CHO': '-.', 'Rezerva': ':'}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, col, naslov in [(axs[0], 'tp_mbps', 'Propusnost [Mbit/s] (više = bolje)'),
                        (axs[1], 'pp10_pct', 'Ping-pong uz prag 10 s [%] (manje = bolje)'),
                        (axs[2], 'rlf_per_min', 'Prekidi veze RLF [1/min] (manje = bolje)')]:
    for m in ['A3', 'D1-CHO', 'Rezerva']:
        ax.axhline(ref.loc[m, col], color=BOJE[m], ls=STIL[m], lw=1.6, label=m)
    ax.plot(ai.trosak, ai[col], '-o', color=BOJE['AI-CHO'], lw=2, ms=6, mec='white', mew=1.2, label='AI-CHO', zorder=3)
    for c_sel, mk, naziv in [(CFG['dp_ho_cost'], 'o', 'stabilna točka'), (CFG['dp_ho_cost_U'], 's', 'uravnotežena točka')]:
        sel = ai[ai.trosak == c_sel]
        ax.scatter(sel.trosak, sel[col], s=150, marker=mk, facecolor='none', edgecolor='#1d3557', linewidth=2, zorder=4,
                   label=f'{naziv} ({c_sel:.0f})')
    ax.set_xscale('log'); ax.set_xticks(KOMPROMIS_TROSKOVI); ax.set_xticklabels([f'{c:.0f}' for c in KOMPROMIS_TROSKOVI])
    ax.minorticks_off(); ax.set_xlabel('trošak primopredaje u pravilu označavanja')
    ax.set_title(naslov, fontsize=10); ax.grid(alpha=.3)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=6, frameon=False, fontsize=9)
fig.suptitle('Kompromis stabilnosti i propusnosti AI-CHO ovisno o trošku primopredaje (validacijski scenarij)', fontsize=11)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.savefig(f'{OUT}/slika_6_3_kompromis.png'); plt.show()

In [ ]:
# Model za uravnoteženu radnu točku, treniran na istim podacima kao glavni model
cfg_U = dict(CFG, dp_ho_cost=CFG['dp_ho_cost_U'])
parts_U = [collect_dataset(cfg_U, tr_) for tr_ in train_traces]
n_train_U = sum(len(p[0]) for p in parts_U)
t1 = time.time()
model_U = HistGradientBoostingClassifier(**MODEL_PARAMS).fit(
    np.vstack([p[0] for p in parts_U]), np.concatenate([p[1] for p in parts_U]), sample_weight=np.concatenate([p[3] for p in parts_U]))
fit_s_U = time.time() - t1
MODELI = {'AI-CHO': model, 'AI-CHO-U': model_U}
CFG_M = {'AI-CHO': CFG, 'AI-CHO-U': cfg_U}
print(f"model AI-CHO-U istreniran (trošak {CFG['dp_ho_cost_U']:.0f}), broj stabala: {model_U.n_iter_}")
del train_traces, tr_fit, tr_val, parts_U

## 5.4. Vrednovanje na neovisnim scenarijima

In [ ]:
import gc
def ml_metrike(mdl, Xt, yt, gt, s):
    p = mdl.predict_proba(Xt)[:, 1]
    pred_dec = pd.Series(p).groupby(gt).idxmax().values; true_dec = pd.Series(yt).groupby(gt).idxmax().values
    is_serv = Xt[:, FEATURES.index('is_serving')] == 1
    sw_true, sw_pred = ~is_serv[true_dec], ~is_serv[pred_dec]
    return dict(seed=s, auc=roc_auc_score(yt, p), row_f1=f1_score(yt, p > 0.5),
                decision_acc=np.mean(pred_dec == true_dec),
                switch_recall=np.mean(pred_dec[sw_true] == true_dec[sw_true]),
                stay_acc=np.mean(pred_dec[~sw_true] == true_dec[~sw_true]),
                switch_precision=np.mean(pred_dec[sw_pred] == true_dec[sw_pred]) if sw_pred.any() else np.nan,
                switch_share_true=np.mean(sw_true), switch_share_pred=np.mean(sw_pred))

results, examples = [], {}
ml_eval = {m: [] for m in MODELI}; timing = {m: [] for m in MODELI}
PAROVI = {'AI-CHO': [], 'AI-CHO-U': []}; N_HO_AI = {'AI-CHO': 0, 'AI-CHO-U': 0}
for s in TEST_SEEDS:
    tr = generate_trace(CFG, s)
    for ai_m, mdl in MODELI.items():                            # idealne odluke prema trošku svake radne točke
        Xt, yt, gt, _ = collect_dataset(CFG_M[ai_m], tr)
        ml_eval[ai_m].append(ml_metrike(mdl, Xt, yt, gt, s))
    for m in METODE:
        df, Ls = evaluate(CFG, tr, m, MODELI.get(m), timing.get(m), return_links=True)
        df['method'] = m; df['seed'] = s; results.append(df)
        if m == 'A3': Ls_a3 = Ls
        if m in PAROVI: PAROVI[m] += upari_ai_a3(Ls, Ls_a3, s); N_HO_AI[m] += sum(len(L.ho_events) for L in Ls)
        if s == TEST_SEEDS[0]:
            for L in Ls: L.tr = None                         # čuvamo samo zapise veze, ne cijeli trag
            examples[m] = Ls
    del tr, Xt, yt, gt, Ls; gc.collect()                     # oslobađanje memorije prije sljedećeg scenarija
print(f"ukupno vrijeme: {(time.time()-t_start)/60:.1f} min")
RES = pd.concat(results, ignore_index=True)
RES.to_csv(f'{OUT}/rezultati_po_korisniku.csv', index=False)

## Rezultati za poglavlje 6
### 6.1. Točnost i računalna učinkovitost modela

In [ ]:
def sazetak_modela(m, fit_t, n_red):
    ML = pd.DataFrame(ml_eval[m]); inf_ms = 1e3*np.array(timing[m])
    return pd.Series({
        'AUC (redak = kandidat)': ML['auc'].mean(),
        'F1 (redak = kandidat)': ML['row_f1'].mean(),
        'Točnost odluke (odabran isti satelit kao idealna odluka)': ML['decision_acc'].mean(),
        'Uravnotežena točnost odluke': 0.5*(ML['switch_recall'].mean() + ML['stay_acc'].mean()),
        'Odziv za odluke o primopredaji': ML['switch_recall'].mean(),
        'Preciznost za odluke o primopredaji': ML['switch_precision'].mean(),
        'Udio odluka za primopredaju: idealno [%]': 100*ML['switch_share_true'].mean(),
        'Udio odluka za primopredaju: model [%]': 100*ML['switch_share_pred'].mean(),
        'Vrijeme zaključivanja po odluci, medijan [ms]': np.median(inf_ms),
        'Vrijeme zaključivanja po odluci, 95. percentil [ms]': np.percentile(inf_ms, 95),
        'Vrijeme treniranja [s]': fit_t,
        'Broj redaka za treniranje': n_red})
summary_ml = pd.DataFrame({f'AI-CHO (trošak {CFG["dp_ho_cost"]:.0f})': sazetak_modela('AI-CHO', fit_s, X_train.shape[0]),
                           f'AI-CHO-U (trošak {CFG["dp_ho_cost_U"]:.0f})': sazetak_modela('AI-CHO-U', fit_s_U, n_train_U)}).round(3)
summary_ml.to_csv(f'{OUT}/tablica_6_1_model.csv')
print(summary_ml.to_string())

In [ ]:
# Važnost značajki mjeri se na razini odluke: koliko pada uravnotežena točnost odluke
# (prosjek točnosti za odluke „ostani” i odziva za odluke o primopredaji) kad se značajka nasumično ispremiješa.
# Značajke izvedene iz iste geometrije jako su korelirane, pa se uz pojedinačne mjeri i važnost po skupinama.
# Važnost se računa za model uravnotežene radne točke (AI-CHO-U), koji se prikazuje na slikama.
GRUPE = {'geometrija': ['elev', 'elev_rate', 'slant_km', 'range_rate'], 'vidljivost': ['rem_vis', 'serv_rem_vis'],
         'mjerenja RSRP': ['rsrp', 'rsrp_trend', 'rsrp_minus_serv', 'serv_rsrp_trend'], 'status poslužnog': ['is_serving'],
         'povijest primopredaje': ['is_prev_serving', 't_since_ho']}

tr = generate_trace(CFG, TEST_SEEDS[0], verbose=False); examples['trace'] = tr
Xt, yt, gt, _ = collect_dataset(CFG_M['AI-CHO-U'], tr)
# Za brzinu se koriste sve odluke o primopredaji i nasumičan uzorak od 4000 odluka „ostani”.
_true = pd.Series(yt).groupby(gt).idxmax().values
_sw = Xt[_true, FEATURES.index('is_serving')] == 0
_rng = np.random.default_rng(0)
_keep = np.concatenate([np.where(_sw)[0], _rng.choice(np.where(~_sw)[0], size=min(4000, (~_sw).sum()), replace=False)])
_mask = np.isin(gt, _keep)
Xd_imp, yd_imp, gd_imp = Xt[_mask], yt[_mask], gt[_mask]

def balanced_decision_acc(Xd, yd, gd):
    p = model_U.predict_proba(Xd)[:, 1]
    pred = pd.Series(p).groupby(gd).idxmax().values; true = pd.Series(yd).groupby(gd).idxmax().values
    sw = Xd[true, FEATURES.index('is_serving')] == 0
    return 0.5*(np.mean(pred[sw] == true[sw]) + np.mean(pred[~sw] == true[~sw]))

def perm_drop(cols, reps=3):
    base_s = balanced_decision_acc(Xd_imp, yd_imp, gd_imp); rng = np.random.default_rng(0); drops = []
    for _ in range(reps):
        Xp = Xd_imp.copy(); idx = rng.permutation(len(Xd_imp))
        for c in cols: Xp[:, FEATURES.index(c)] = Xd_imp[idx, FEATURES.index(c)]
        drops.append(base_s - balanced_decision_acc(Xp, yd_imp, gd_imp))
    return np.mean(drops)

IMP = pd.Series({f: perm_drop([f]) for f in FEATURES}).sort_values()
IMP_G = pd.Series({g: perm_drop(c) for g, c in GRUPE.items()}).sort_values()
IMP.to_csv(f'{OUT}/vaznost_znacajki.csv', header=['pad_uravnotezene_tocnosti'])
IMP_G.to_csv(f'{OUT}/vaznost_skupina_znacajki.csv', header=['pad_uravnotezene_tocnosti'])
print(IMP_G.round(3).to_string())
from matplotlib.ticker import MaxNLocator
def crtaj_vaznost(ser, boja, naslov, datoteka, visina):
    fig, ax = plt.subplots(figsize=(7.5, visina))
    ax.barh(ser.index, ser.values, color=boja)
    ax.axvline(0, color='#555', lw=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))        # manje oznaka, da se brojevi s minusom ne preklapaju
    ax.set_xlabel('pad uravnotežene točnosti odluke'); ax.set_title(naslov); ax.grid(axis='x', alpha=.3)
    plt.tight_layout(); plt.savefig(f'{OUT}/{datoteka}'); plt.show()

crtaj_vaznost(IMP_G, BOJE['AI-CHO-U'], 'Važnost skupina značajki (AI-CHO-U)', 'slika_6_1_vaznost_skupina_znacajki.png', 3.0)

### 6.2. i 6.3. Usporedba metoda primopredaje

In [ ]:
METRIKE = {'ho_per_min': 'Primopredaje po korisniku [1/min]', 'pingpong_pct': 'Ping-pong primopredaje [%]',
           'pp10_pct': 'Ping-pong uz prag 10 s [%]', 'pp30_pct': 'Ping-pong uz prag 30 s [%]',
           'rlf_per_min': 'Prekidi veze RLF po korisniku [1/min]', 'hof_per_min': 'Neuspjele primopredaje HOF [1/min]',
           'outage_pct': 'Udio vremena bez upotrebljive veze [%]', 'tp_mbps': 'Prosječna propusnost [Mbit/s]',
           'mean_snr_db': 'Prosječni SNR [dB]'}
per_seed = RES.groupby(['method', 'seed'])[list(METRIKE)].mean()
TAB = per_seed.groupby('method').agg(['mean', 'std']).reindex(METODE)
nice = pd.DataFrame({METRIKE[m]: TAB[m]['mean'].round(3).astype(str) + ' ± ' + TAB[m]['std'].round(3).astype(str)
                     for m in METRIKE}).T
nice.to_csv(f'{OUT}/tablica_6_2_usporedba.csv')
print(nice.to_string())

### Statistička značajnost razlika
Sve metode izvode se nad istim kanalnim tragom, pa su rezultati po scenarijima upareni. Za svaku metriku
uspoređuju se obje radne točke AI-CHO s A3, D1-CHO i ablacijom (Rezerva) Wilcoxonovim testom predznačenih rangova (dvostrani test).
Uz 10 scenarija najmanja moguća p-vrijednost je oko 0,002; razlika se smatra značajnom za p < 0,05.

In [ ]:
from scipy.stats import wilcoxon
rows = []
for m in METRIKE:
    a = per_seed[m].unstack(0)
    for ai_m in ['AI-CHO', 'AI-CHO-U']:
      for ref in ['A3', 'D1-CHO', 'Rezerva']:
        d = a[ai_m] - a[ref]
        p = wilcoxon(a[ai_m], a[ref]).pvalue if np.any(d != 0) else 1.0
        rows.append(dict(metrika=METRIKE[m], usporedba=f'{ai_m} − {ref}', srednja_razlika=d.mean(),
                         bolji_u_scenarija=f"{int((d < 0).sum() if m in ['ho_per_min','pingpong_pct','pp10_pct','pp30_pct','rlf_per_min','hof_per_min','outage_pct'] else (d > 0).sum())}/{len(d)}",
                         p=p))
SIG = pd.DataFrame(rows); SIG['znacajno'] = np.where(SIG.p < 0.05, 'da', 'ne')
SIG.round(4).to_csv(f'{OUT}/tablica_6_3_znacajnost.csv', index=False)
print(SIG.round(4).to_string(index=False))

In [ ]:
by_type = RES.assign(tip=np.where(RES.moving, 'u vozilu', 'miruje')).groupby(['tip', 'method'])[
    ['ho_per_min', 'pingpong_pct', 'rlf_per_min', 'tp_mbps']].mean().round(3)
by_type.to_csv(f'{OUT}/dodatak_po_tipu_korisnika.csv'); print(by_type)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4.0))
PRIKAZ = ['A3', 'D1-CHO', 'Rezerva', 'AI-CHO-U']              # AI-CHO (trošak 20) prikazan je u tablici 6.2
for ax, m in zip(axs, ['ho_per_min', 'pp10_pct', 'rlf_per_min', 'tp_mbps']):
    mu, sd = TAB[m]['mean'].reindex(PRIKAZ), TAB[m]['std'].reindex(PRIKAZ)
    ax.bar(mu.index, mu.values, yerr=sd.values, capsize=4, color=[BOJE[i] for i in mu.index])
    ax.set_title(METRIKE[m], fontsize=10); ax.grid(axis='y', alpha=.3)
    ax.tick_params(axis='x', labelrotation=30, labelsize=9)
    for lab in ax.get_xticklabels(): lab.set_ha('right')
plt.tight_layout(); plt.savefig(f'{OUT}/slika_6_2_usporedba_metoda.png'); plt.show()

## Grafovi za poglavlja 3 i 5

In [ ]:
# Slika 3.4: Dopplerov pomak i kašnjenje tijekom jednog preleta (satelit s najvećom elevacijom za korisnika 0)
tr = examples['trace']; tt = np.arange(tr['T'])*tr['dt']
j = int(np.argmax(tr['elev'][:, 0].max(0))); e = tr['elev'][:, 0, j]; vis = e >= 0
d = np.sqrt((RE*np.sin(np.radians(e)))**2 + CFG['h_km']**2 + 2*CFG['h_km']*RE) - RE*np.sin(np.radians(e))
dop = -tr['rr'][:, 0, j]/C_KMS*CFG['fc_hz']/1e3
fig, ax1 = plt.subplots(figsize=(7.5, 4)); ax2 = ax1.twinx()
ax1.plot(tt[vis], dop[vis], color='#2E6DA4', lw=2, label='Dopplerov pomak')
ax2.plot(tt[vis], d[vis]/C_KMS*1e3, color='#E07A2F', lw=2, label='kašnjenje u jednom smjeru')
ax1.set_xlabel('vrijeme [s]'); ax1.set_ylabel('Dopplerov pomak [kHz]', color='#2E6DA4')
ax2.set_ylabel('kašnjenje [ms]', color='#E07A2F'); ax1.grid(alpha=.3)
ax1.set_title(f'Prelet satelita na {CFG["h_km"]:.0f} km (najveća elevacija {e.max():.0f}°, {CFG["fc_hz"]/1e9:.0f} GHz)')
plt.tight_layout(); plt.savefig(f'{OUT}/slika_3_4_doppler_kasnjenje.png'); plt.show()

In [ ]:
# Slika 3.5: RSRP izvornog i ciljnog satelita oko jedne primopredaje (metoda A3, korisnik 0)
L = examples['A3'][0]
if L.ho_times:
    th = L.ho_times[len(L.ho_times)//2]; kh = int(th/tr['dt'])
    s_old, s_new = L.serv_log[kh-2], L.serv_log[kh+2]
    w = slice(max(kh-300, 0), min(kh+300, tr['T']))
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.plot(tt[w], tr['rsrp_meas'][w, 0, s_old], color='#2E6DA4', lw=2, label='izvorni satelit')
    ax.plot(tt[w], tr['rsrp_meas'][w, 0, s_new], color='#E07A2F', lw=2, label='ciljni satelit')
    ax.axvline(th, color='grey', ls='--'); ax.text(th+0.5, ax.get_ylim()[1]-2, 'primopredaja', color='grey')
    ax.set_xlabel('vrijeme [s]'); ax.set_ylabel('filtrirani RSRP [dBm]'); ax.legend(); ax.grid(alpha=.3)
    ax.set_title('Snaga signala dvaju satelita oko primopredaje')
    plt.tight_layout(); plt.savefig(f'{OUT}/slika_3_5_rsrp_primopredaja.png'); plt.show()

### Primopredaje koje je odlučio AI model, uspoređene s A3
Sve metode rade nad istim kanalnim tragom, pa se za svaku primopredaju (svih 10 testnih scenarija) koju je pokrenuo **sam model**
(ne rezervni okidač) traži primopredaja istog korisnika između **istih dvaju satelita** kod metode A3,
unutar ±60 s. Razlika vremena (A3 − AI) pokazuje koliko ranije ili kasnije model reagira.
Statistika se računa za obje radne točke (tablica 6.5), a slike 6.4 i 6.5 crtaju se za AI-CHO-U.
Primjer na slici odabran je automatski: to je usporedba s medijanom te razlike, a ne ručno odabran slučaj.

In [ ]:
PROZOR_S = 60.0
OZNAKE = {'AI-CHO': 'AI-CHO, trošak %.0f' % CFG['dp_ho_cost'], 'AI-CHO-U': 'AI-CHO-U, trošak %.0f' % CFG['dp_ho_cost_U']}
CRTAJ = 'AI-CHO-U'                                               # slike samo za uravnoteženu točku, obje u tablici 6.5
PAR_SVE, SAZ = [], []
for ai_m, naziv in OZNAKE.items():
    PAR = pd.DataFrame(PAROVI[ai_m]); PAR['prednost_s'] = PAR.t_a3 - PAR.t_ai; PAR['metoda'] = ai_m
    uparene = PAR.dropna(); PAR_SVE.append(PAR)
    print(f"--- {naziv} ---")
    print(f"primopredaje ukupno (10 scenarija): {N_HO_AI[ai_m]}, od toga odlukom modela: {len(PAR)} "
          f"({100*len(PAR)/max(N_HO_AI[ai_m],1):.0f} %)")
    print(f"uparene s istom primopredajom kod A3: {len(uparene)}; model ranije u {100*np.mean(uparene.prednost_s > 0):.0f} % slučajeva")
    print(f"razlika A3 − AI: medijan {uparene.prednost_s.median():.1f} s, "
          f"interkvartilni raspon {uparene.prednost_s.quantile(.25):.1f} do {uparene.prednost_s.quantile(.75):.1f} s")
    pr = uparene.prednost_s
    udjeli = (100*(pr.abs() <= 2).mean(), 100*(pr > 2).mean(), 100*(pr < -2).mean())
    print(f"istovremeno (±2 s): {udjeli[0]:.0f} %, model ranije > 2 s: {udjeli[1]:.0f} % (medijan {pr[pr > 2].median():.1f} s), "
          f"model kasnije > 2 s: {udjeli[2]:.0f} %")
    SAZ.append({'metoda': naziv, 'primopredaje ukupno': N_HO_AI[ai_m], 'odlukom modela [%]': 100*len(PAR)/max(N_HO_AI[ai_m], 1),
                'uparene s A3': len(uparene), 'istovremeno ±2 s [%]': udjeli[0], 'model ranije > 2 s [%]': udjeli[1],
                'medijan prednosti kad je ranije [s]': pr[pr > 2].median(), 'model kasnije > 2 s [%]': udjeli[2]})
    if ai_m != CRTAJ: continue

    # Slika 6.5(b): raspodjela vremenske razlike (pozitivno = model prebacuje ranije od A3)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.hist(uparene.prednost_s, bins=np.arange(-PROZOR_S, PROZOR_S + 1, 4), color=BOJE[ai_m], edgecolor='white')
    ax.axvline(0, color='#555', lw=1)
    ax.axvline(uparene.prednost_s.median(), color='#1d3557', ls='--', lw=1.5)
    ax.text(0.98, 0.95, f'medijan {pr.median():.1f} s\nistovremeno (±2 s): {udjeli[0]:.0f} %\n'
            f'model ranije (> 2 s): {udjeli[1]:.0f} %\nmodel kasnije (> 2 s): {udjeli[2]:.0f} %',
            transform=ax.transAxes, ha='right', va='top', fontsize=8.5, color='#1d3557',
            bbox=dict(facecolor='white', edgecolor='#ccc', boxstyle='round,pad=0.4'))
    ax.set_xlabel(f't(A3) − t({ai_m}) [s]   (pozitivno = model prebacuje ranije)'); ax.set_ylabel('broj primopredaja')
    ax.set_title(f'Vremenska razlika između odluke modela i iste primopredaje kod A3 ({naziv})', fontsize=10)
    ax.grid(axis='y', alpha=.3)
    plt.tight_layout(); plt.savefig(f'{OUT}/slika_6_4_razlika_ai_a3.png'); plt.show()

    # Slika 6.6(b): primjer najbliži medijanu, iz scenarija čiji je kanalni trag spremljen
    prim = uparene[uparene.seed == TEST_SEEDS[0]]
    if not len(prim): continue
    r = prim.iloc[(prim.prednost_s - uparene.prednost_s.median()).abs().argmin()]
    u, a, b = int(r.korisnik), int(r.izvor), int(r.cilj)
    t_lo, t_hi = min(r.t_ai, r.t_a3) - 30, max(r.t_ai, r.t_a3) + 30
    w = slice(max(int(t_lo/tr['dt']), 0), min(int(t_hi/tr['dt']), tr['T']))
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    ispod = tr['snr'][w, u, a] < CFG['q_out_db']
    ax.plot(tt[w], tr['rsrp_meas'][w, u, a], color='#555', lw=2, label='izvorni satelit')
    ax.plot(tt[w], tr['rsrp_meas'][w, u, b], color='#9EC5E8', lw=2, label='ciljni satelit')
    y_lo, y_hi = ax.get_ylim()                          # raspon iz krivulja; sjenčanje ga ne smije mijenjati
    if ispod.any():
        ax.fill_between(tt[w], 0, 1, where=ispod, transform=ax.get_xaxis_transform(), color='#C0392B', alpha=.15, lw=0,
                        label='izvorni satelit ispod praga za prekid veze')
    ax.axvline(r.t_ai, color=BOJE[ai_m], lw=2, label=f'odluka {ai_m} ({r.t_ai:.1f} s)')
    ax.axvline(r.t_a3, color=BOJE['A3'], lw=2, ls='--', label=f'primopredaja A3 ({r.t_a3:.1f} s)')
    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel('vrijeme [s]'); ax.set_ylabel('filtrirani RSRP [dBm]'); ax.grid(alpha=.3)
    ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    ax.set_title(f'{naziv}: odluka modela naspram A3 (primjer: razlika {r.prednost_s:.1f} s; '
                 f'medijan svih {uparene.prednost_s.median():.1f} s)', fontsize=10)
    plt.tight_layout(); plt.savefig(f'{OUT}/slika_6_5_primjer_ai_primopredaje.png'); plt.show()

pd.concat(PAR_SVE).round(2).to_csv(f'{OUT}/dodatak_parovi_ai_a3.csv', index=False)
TAB65 = pd.DataFrame(SAZ).set_index('metoda').T.round(1); TAB65.to_csv(f'{OUT}/tablica_6_5_ai_naspram_a3.csv'); print(TAB65.to_string())

In [ ]:
# Slika 5.1: trenutni položaj konstelacije i vidljivi sateliti iznad ruralnog područja
rs, _ = sat_state_ecef(tr['t0'])
lat = np.degrees(np.arcsin(rs[:, 2]/np.linalg.norm(rs, axis=1))); lon = np.degrees(np.arctan2(rs[:, 1], rs[:, 0]))
ru = geodetic_to_ecef(np.array([CFG['lat0']]), np.array([CFG['lon0']]))
e0, *_ = geometry(rs, np.zeros_like(rs), ru, np.zeros_like(ru)); visn = e0[0] >= CFG['elev_min_deg']
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(lon, lat, s=4, color='#9EC5E8', label='sateliti')
ax.scatter(lon[visn], lat[visn], s=18, color='#E07A2F', label=f'vidljivi iznad {CFG["elev_min_deg"]:.0f}° ({visn.sum()})')
ax.scatter([CFG['lon0']], [CFG['lat0']], marker='*', s=200, color='#1d3557', label='ruralno područje')
ax.set_xlim(-180, 180); ax.set_ylim(-60, 60); ax.set_xlabel('zemljopisna dužina [°]'); ax.set_ylabel('zemljopisna širina [°]')
ax.set_title(f'Walker-delta {CFG["incl_deg"]:.0f}°: {N_SATS}/{CFG["P"]}/{CFG["F"]}, h = {CFG["h_km"]:.0f} km')
ax.legend(loc='lower left', fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/slika_5_1_konstelacija.png'); plt.show()

In [ ]:
# Slika 5.2: poslužni satelit kroz vrijeme za korisnika 0 kod svake metode
fig, axs = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, m in zip(axs, ['A3', 'D1-CHO', 'AI-CHO-U']):
    sl = examples[m][0].serv_log.astype(float); sl[sl < 0] = np.nan
    ax.step(tt, sl, where='post', color=BOJE[m], lw=1.5); ax.set_ylabel(m, rotation=0, ha='right', va='center')
    ax.set_yticks([]); ax.grid(alpha=.3)
axs[0].set_title('Poslužni satelit kroz vrijeme (svaka razina = jedan satelit, prekid = bez veze)')
axs[-1].set_xlabel('vrijeme [s]'); plt.tight_layout(); plt.savefig(f'{OUT}/slika_5_2_posluzni_satelit.png'); plt.show()

## Spremanje svih rezultata
Mapa `rezultati` sadrži tablice (CSV), slike (PNG, 300 dpi), parametre (`parametri.json`) i inačice biblioteka (`verzije.txt`).
Ćelija ispod ih sprema i pakira u ZIP za preuzimanje.

In [ ]:
json.dump({k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in CFG.items()},
          open(f'{OUT}/parametri.json', 'w'), indent=2, default=str)
# Inačice okruženja (za Prilog A): spremaju se uz rezultate u verzije.txt
import sys, platform, sklearn, scipy, matplotlib
verzije = {'Python': platform.python_version(), 'NumPy': np.__version__, 'pandas': pd.__version__,
           'scikit-learn': sklearn.__version__, 'SciPy': scipy.__version__, 'Matplotlib': matplotlib.__version__}
with open(f'{OUT}/verzije.txt', 'w') as f:
    for k, v in verzije.items(): f.write(f'{k}: {v}\n')
print(open(f'{OUT}/verzije.txt').read())
import shutil
shutil.make_archive('rezultati', 'zip', OUT)
try:
    from google.colab import files; files.download('rezultati.zip')
except ImportError:
    print('Rezultati su spremljeni u rezultati.zip')